In [0]:
tickets = spark.read.format("json").option("multiline","true").option("mode","PERMISSIVE").load("/mnt/jspl_datalake/GI_API_RAW_DEV/fresh_desk/tickets/*/*.json")

In [0]:
contacts = spark.read.format("json").option("multiline","true").option("mode","PERMISSIVE").load("/mnt/jspl_datalake/GI_API_RAW_DEV/fresh_desk/contacts/*/*.json")

In [0]:
agents = spark.read.format("json").option("multiline","true").option("mode","PERMISSIVE").load("/mnt/jspl_datalake/GI_API_RAW_DEV/fresh_desk/agents/*/*.json")

In [0]:
companies = spark.read.format("json").option("multiline","true").option("mode","PERMISSIVE").load("/mnt/jspl_datalake/GI_API_RAW_DEV/fresh_desk/companies/*/*.json")

In [0]:
from pyspark.sql.functions import col

tickets_flat = tickets.select(

col("id").alias("ticket_id"),
col("created_at"),
col("stats.closed_at").alias("closed_date"),

col("subject"),
col("status"),
col("priority"),
col("source"),

col("requester_id"),
col("responder_id"),
col("company_id"),

col("custom_fields.cf_po_number").alias("po_number"),
col("custom_fields.cf_invoice_number").alias("invoice_number"),
col("custom_fields.cf_vendor_code").alias("vendor_code"),
col("custom_fields.cf_nimble_task_id").alias("nimble_task_id"),
col("custom_fields.cf_fetch_using_task_id").alias("task_id"),
col("custom_fields.cf_vendor_name").alias("vendor_name"),
col("custom_fields.cf_query_type").alias("query_type"),
col("custom_fields.cf_location").alias("location"),
col("custom_fields.cf_company_code").alias("company_code")

)

In [0]:
agents_dim = agents.select(
col("id").alias("agent_id"),
col("contact.name").alias("agent_name")
)

In [0]:
contacts_dim = contacts.select(
col("id").alias("contact_id"),
col("email").alias("requester_email")
)

In [0]:
companies_dim = companies.select(
col("id").alias("company_id"),
col("name").alias("company_name")
)

In [0]:
df = tickets_flat \
    .join(agents_dim.dropDuplicates(["agent_id"]),
          tickets_flat.responder_id == agents_dim.agent_id,
          "left") \
    .join(contacts_dim.dropDuplicates(["contact_id"]),
          tickets_flat.requester_id == contacts_dim.contact_id,
          "left") \
    .join(companies_dim.dropDuplicates(["company_id"]),
          "company_id",
          "left")

In [0]:
from pyspark.sql.functions import datediff

df = df.withColumn(
"resolution_days",
datediff("closed_date","created_at")
)

In [0]:
from pyspark.sql.functions import month

df = df.withColumn(
"month",
month("created_at")
)

In [0]:
final_df = df.select(

"ticket_id",
"created_at",
"closed_date",
"resolution_days",
"agent_name",
"month",
"location",
"query_type",
"status",
"priority",
"source",
"subject",
"po_number",
"invoice_number",
"vendor_code",
"nimble_task_id",
"vendor_name",
"company_name",
"company_code",
"requester_email"

)

In [0]:
final_df = final_df.distinct()

In [0]:
final_df.count()

58987

In [0]:
final_df.write.mode("overwrite").saveAsTable("gi_gold.help_desktickets")

In [0]:
output_path = "dbfs:/mnt/jspl_datalake/GI_SAP_TRANSFORMED_DEV/help_desktickets"

final_df.write \
   .format("csv") \
   .option("header", "true") \
   .mode("overwrite") \
   .save(output_path)